In [11]:
NUM_ANTS = 2                  # number of ants in the colony (i hate ants!!!!)
EVAPORATION = 0.2               # how much pheromone evaporates each iteration
ITERATIONS = 2                # number of iterations to run the algorithm
FEATURES_PER_ITERATION = 3    # number of features to select per iteration
BETA = 0.5                      # weight given for similarity vs pheromone (beta > 1, the heuristic dominates)
EXPLOITATION_RATE = 0.8         # how much to exploit vs explore
EPSILON = 0.00001               # Stop division by zero errors
SIMILARITY_FUNCTION = 'phi'     # similarity function to use, 'phi' or 'mi'
SAVE_PHEROMONES = True          # whether to save pheromone matrix after each iteration
LOAD_PHEROMONES = False         # whether to load existing pheromone matrix to resume

from config import IMPUTATION   # whether to use imputed values

In [12]:
from urielplus import urielplus
from sklearn.metrics import matthews_corrcoef, normalized_mutual_info_score
import numpy as np
import pandas as pd
from tqdm import tqdm
import os
from fancyimpute import SoftImpute

from langrank import *

In [13]:
# cursor said this shuts up SoftImpute
import warnings
warnings.filterwarnings("ignore", category=FutureWarning, message="'force_all_finite'")

# shut up LightGBM
warnings.filterwarnings('ignore', category=UserWarning, module='lightgbm')

# Initialization

In [14]:
uriel = urielplus.URIELPlus()
uriel.integrate_databases()
uriel.set_aggregation('U')
uriel.aggregate()

2025-08-08 15:09:45,860 - root - INFO - Importing all databases....
2025-08-08 15:09:45,863 - root - INFO - Importing updated SAPHON from "saphon_data.csv"....
2025-08-08 15:09:46,860 - root - INFO - Updated SAPHON integration complete..
2025-08-08 15:09:46,862 - root - INFO - Importing BDPROTO from "bdproto_data.csv"....
2025-08-08 15:09:46,864 - root - INFO - Converting ISO 639-3 codes to Glottocodes....
2025-08-08 15:09:47,020 - root - INFO - Conversion to Glottocodes complete.
2025-08-08 15:09:59,126 - root - INFO - BDPROTO integration complete.
2025-08-08 15:09:59,129 - root - INFO - Importing Grambank from "grambank_data.csv"....
2025-08-08 15:10:33,985 - root - INFO - Grambank integration complete.
2025-08-08 15:10:33,987 - root - INFO - Importing APiCS from "apics_data.csv"....
2025-08-08 15:10:37,056 - root - INFO - APiCS integration complete.
2025-08-08 15:10:37,058 - root - INFO - Importing eWAVE from "english_dialect_data.csv"....
2025-08-08 15:10:44,993 - root - INFO - eWA

array([[[-1.],
        [-1.],
        [-1.],
        ...,
        [-1.],
        [-1.],
        [-1.]],

       [[-1.],
        [-1.],
        [-1.],
        ...,
        [-1.],
        [-1.],
        [-1.]],

       [[-1.],
        [-1.],
        [-1.],
        ...,
        [-1.],
        [-1.],
        [-1.]],

       ...,

       [[ 1.],
        [ 0.],
        [ 0.],
        ...,
        [ 1.],
        [ 1.],
        [ 1.]],

       [[ 1.],
        [ 0.],
        [ 0.],
        ...,
        [ 1.],
        [ 1.],
        [ 1.]],

       [[ 1.],
        [ 0.],
        [ 0.],
        ...,
        [ 0.],
        [ 1.],
        [ 1.]]])

In [15]:
# Collect aggregated data
data: np.ndarray = np.squeeze(uriel.get_typological_data_array())
data.shape

(8174, 800)

# Similarity Functions

In [16]:
def phi_coefficient(x: np.ndarray, y: np.ndarray) -> float:
    """
    Calculate the absolute value of Matthews Correlation Coefficient (phi coefficient) for two binary vectors.

    Input
    -----
    - x: array-like (num_features,)
    - y: array-like (num_features,)

    Output
    ------
    The absolute value of the phi coefficient
    """
    return abs(matthews_corrcoef(x, y))

In [17]:
def mutual_information(x: np.ndarray, y: np.ndarray) -> np.float64:
    """
    Calculate the normalized mutual information between two binary vectors.

    Input
    -----
    - x: array-like (num_features,)
    - y: array-like (num_features,)

    Output
    ------
    The normalize mutual information score between x and y.
    """
    return normalized_mutual_info_score(x, y)

# Helper Functions

In [18]:
def construct_weight_matrix(data: np.ndarray, similarity_function) -> np.ndarray:
    """
    Construct a weight (similarity) matrix from the data.

    Input
    -----
    - data: array-like (num_samples, num_features)
    - similarity_function: function to compute similarity between two features, follows the signature; func([ndarray], [ndarray]) -> float
    
    Output
    ------
    Square weight matrix of shape (num_features, num_features) where Wij is the similarity between feature i and feature j. 
    Wii is set to the maximum similarity value.
    """
    num_features = data.shape[1]
    weights = np.ones((num_features, num_features))
    
    for i in range(num_features):
        for j in range(i):
            weights[i, j] = similarity_function(data[:, i], data[:, j])
            weights[j, i] = weights[i, j]
    
    return weights

np.ones((data.shape[1], data.shape[1])).shape

(800, 800)

In [19]:
def construct_pheromone_matrix(data: np.ndarray) -> np.ndarray:
    """
    Construct a pheromone matrix from the data.

    Input
    -----
    - data: array-like, shape (n_samples, n_features)

    Output
    ------
    - pheromones: array-like, shape (n_features,)
    """
    num_features = data.shape[1]
    
    # Try to load existing pheromones if requested
    pheromone_file = f'pheromones_{SIMILARITY_FUNCTION}_{NUM_ANTS}_{FEATURES_PER_ITERATION}.npy'
    if LOAD_PHEROMONES:
        pheromones = np.load(pheromone_file)
        return pheromones

    pheromones = np.ones((num_features,))
    
    return pheromones

construct_pheromone_matrix(data).shape

(800,)

In [33]:
def compute_pheromones(data: np.ndarray, selected_features: set[int]) -> float:
    """
    Compute the pheromones for a given subset of features evaluated on LangRank.

    Input
    -----
    - data: array-like, shape (n_samples, n_features), the full URIEL dataset before feature selection
    - selected_features: set of feature indices that are selected

    Output
    ------
    - loss: float, the loss value computed from the LangRank results
    """
    FEATURE_TYPES = ['syntactic', 'inventory', 'phonological', 'featural', 'morphological']
    TASKS = {   # Bool is to indicate whether the dataset is in ISO and needs to be converted to glottocode
        'mt': ('BLEU', True),
        'dep': ('accuracy', True),
        'el': ('accuracy', True),
        'pos': ('accuracy', True),
        'taxi1500': ('f1_score', False),
        'xnli': ('accuracy', False),
    }

    script_df = pd.read_csv('data/URIEL_Script.csv', index_col=0)
    islands_df = pd.read_csv('data/URIELPlus_Union_Imputed.csv', index_col=0)
    phylogeny_df = pd.read_csv('data/URIEL_Phylogeny.csv', index_col=0)
    geography_df = pd.read_csv('data/URIEL_Geography.csv', index_col=0)

    subset: np.ndarray = data[:, list(selected_features)]
    subset = np.where(subset == -1, np.nan, subset)

    if IMPUTATION:
        imputer = SoftImpute(max_iters=400,  max_value=1, min_value=0, init_fill_method="mean", verbose=False)
        imputed_values = imputer.fit_transform(subset)

        df = pd.DataFrame(imputed_values, columns=uriel.get_typological_features_array()[np.array(list(selected_features))], index=uriel.get_typological_languages_array())
    else:
        df = pd.DataFrame(subset, columns=uriel.get_typological_features_array()[np.array(list(selected_features))], index=uriel.get_typological_languages_array())

    # baseline = pd.read_csv('data/baseline_results_imputed.csv').to_numpy().squeeze()

    # Create calculators using the data
    calculators: dict[str, DistanceCalculator] = {
        'syntactic': SyntacticCalculator(df),
        'morphological': MorphologicalCalculator(df),
        'inventory': InventoryCalculator(df),
        'phonological': PhonologicalCalculator(df),
        'featural': FeaturalCalculator(df),

        'scriptural': GenericCalculator(script_df),
        'islands': IslandCalculator(islands_df),
        'new_geographic': GeographicCalculator(1),
        'geographic': GenericCalculator(geography_df),
        'genetic': GenericCalculator(phylogeny_df)
    }

    evaluator = LangRankEvaluator(
        calculators = calculators,
        iso_map_file = 'data/code_mapping.csv'
    )

    eval_results = np.zeros(len(TASKS))
    for i, task in enumerate(TASKS):
        df = evaluator.replace_distances(
            data_file = f'data/{task}.csv',
            distance_types = FEATURE_TYPES, 
            iso_conversion = TASKS[task][1]
        )
        
        score = evaluator.evaluate(
            data = df,
            features = FEATURE_TYPES,
            performance_col_name = TASKS[task][0],
        )
        
        eval_results[i] = score[0]

    reward: float = float(np.mean(eval_results))

    return reward

# Unsupervised Feature Selection based on Ant Colony Optimization (UFSACO)
An unsupervised feature selection algorithm based on ant colony optimization (Tabakhi, 2014).

In [21]:
class Ant:
    """
    Class representing an ant in the Ant Colony Optimization algorithm.
    Each ant has a current feature and a set of selected features.
    """
    def __init__(self, initial_feature: int):
        self.current_feature = initial_feature
        self.selected_features: set[int] = set()

In [22]:
def choose_feature(ant: Ant, pheromones: np.ndarray, weights: np.ndarray, mode: str = 'prob') -> int:
    """
    Choose a feature for the ant to select based on pheromone levels and weights.
    
    Inputs
    ------
    - ant: Ant object representing the current ant
    - pheromones: array-like, shape (n_features,)
    - weights: array-like, shape (n_features, n_features)
    - mode: str, either 'prob' for probabilistic selection or 'greedy' for deterministic selection - default 'prob'

    Output
    ------
    The index of the selected feature.
    """
    num_features = pheromones.shape[0]

    if len(ant.selected_features) == num_features:
        raise ValueError("All features have been selected. Ensure that FEATURES_PER_ITERATION is less than the number of features.")

    mask = np.zeros(num_features, dtype=int)
    mask[list(ant.selected_features)] = 1

    logits = pheromones * ((1/(weights[ant.current_feature, :] + EPSILON)) ** BETA)
    logits = np.where(mask == 0, logits, 0)

    if mode == 'prob': 
        probabilities: np.ndarray = logits / np.sum(logits)
        feature: int = np.random.choice(range(num_features), p=probabilities)
    elif mode == 'greedy':        
        feature: int = int(np.argmax(logits))
    else:
        raise ValueError("Invalid mode. Choose 'prob' or 'greedy'.")

    return feature

In [23]:
def select_features_ACO(data: np.ndarray, weights: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """
    Select features using the Ant Colony Optimization algorithm.
    The algorithm iteratively updates pheromone levels based on the features selected by the ants.

    Inputs
    ------
    - data: array-like, shape (n_languages, n_features)
    - weights: array-like, shape (n_features, n_features), the similarity matrix between features

    Outputs
    -------
    - ndarray of indicies of features, ordered by final pheromone levels in descending order.
    - ndarray of pheromone levels, ordered by final pheromone levels in descending order.
    """
    num_features = data.shape[1]

    pheromones: np.ndarray = construct_pheromone_matrix(data)

    for iteration in tqdm(range(ITERATIONS), desc="ACO Progress"):
        # Initial placement of ants (no duplicates), does not count towards feature_count or pheromones
        ants: list[Ant] = []
        placed_features = set()
        for _ in range(NUM_ANTS):
            feature: int = np.random.choice(list(set(range(num_features)) - placed_features))
            ant = Ant(feature)
            placed_features.add(feature)
            ants.append(ant)

        # Iterate
        for feature_choice in range(FEATURES_PER_ITERATION):
            for ant in ants:
                mode: str = 'prob' if np.random.rand() > EXPLOITATION_RATE else 'greedy'
                feature: int = choose_feature(ant, pheromones, weights, mode)
                ant.current_feature = feature
                ant.selected_features.add(feature)

        # Update pheromones
        pheromones *= (1 - EVAPORATION)
        for ant in ants:
            # Only improve pheromones
            reward = compute_pheromones(data, ant.selected_features)
            pheromones[np.array(sorted(ant.selected_features), dtype=int)] += max(0, reward)

        # Save pheromones after each iteration if requested
        if SAVE_PHEROMONES:
            pheromone_file = f'pheromones_{SIMILARITY_FUNCTION}_{NUM_ANTS}_{FEATURES_PER_ITERATION}.npy'
            np.save(pheromone_file, pheromones)
            print(f"Saved pheromone matrix to {pheromone_file}")

    # Sort features by pheromone levels in descending order
    return np.argsort(pheromones)[::-1], np.sort(pheromones)[::-1]

# How to run

In [24]:
data: np.ndarray = np.squeeze(uriel.get_typological_data_array())   # No imputation!
feature_labels: np.ndarray = uriel.get_typological_features_array()
languages: np.ndarray = uriel.get_typological_languages_array()

df = pd.DataFrame(data, columns=feature_labels, index=languages)

In [25]:
similarity_function = phi_coefficient if SIMILARITY_FUNCTION == 'phi' else mutual_information
weights: np.ndarray = construct_weight_matrix(data, similarity_function)

In [34]:
ranked_features, pheromones = select_features_ACO(data, weights)

2862it [00:02, 1220.18it/s]  | 0/2 [00:00<?, ?it/s]
870it [00:00, 1308.15it/s]
477it [00:00, 1363.65it/s]
1545it [00:01, 1408.73it/s]
26334it [00:21, 1234.51it/s]
225it [00:00, 484.37it/s]
2862it [00:02, 1159.88it/s]
870it [00:00, 1093.99it/s]
477it [00:00, 1094.10it/s]
1545it [00:01, 1119.69it/s]
26334it [00:23, 1121.63it/s]
225it [00:00, 756.64it/s]
ACO Progress:  50%|█████     | 1/2 [07:23<07:23, 443.07s/it]

Saved pheromone matrix to pheromones_phi_2_3.npy


2862it [00:02, 1243.34it/s]
870it [00:00, 1004.89it/s]
477it [00:00, 1117.18it/s]
1545it [00:01, 1125.92it/s]
26334it [00:23, 1106.20it/s]
225it [00:00, 599.83it/s]
2862it [00:03, 906.00it/s] 
870it [00:02, 423.89it/s]
477it [00:00, 642.44it/s]
1545it [00:01, 997.40it/s] 
26334it [00:31, 841.15it/s]
225it [00:00, 680.77it/s]
ACO Progress: 100%|██████████| 2/2 [17:54<00:00, 537.20s/it]

Saved pheromone matrix to pheromones_phi_2_3.npy


In [ ]:
# Save results
for num_features in range(100, 701, 100):
    filtered_data: pd.Series = df.iloc[:, ranked_features[:num_features]]

    df_np = filtered_data.to_numpy()
    df_np = np.where(df_np == -1, np.nan, df_np)

    df_final = pd.DataFrame(df_np, columns=filtered_data.columns, index=filtered_data.index)

    if not os.path.exists('selection_result'):
        os.makedirs('selection_result')

    save = f'selection_result/ant_{SIMILARITY_FUNCTION}_{num_features}.csv'
    df_final.to_csv(save)

c:\Users\Chi\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\Chi\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\Chi\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


[SoftImpute] Max Singular Value of X_init = 418.455346
[SoftImpute] Iter 1: observed MAE=0.149896 rank=60
[SoftImpute] Iter 2: observed MAE=0.148261 rank=56
[SoftImpute] Iter 3: observed MAE=0.147225 rank=54
[SoftImpute] Iter 4: observed MAE=0.146567 rank=53
[SoftImpute] Iter 5: observed MAE=0.146134 rank=52
[SoftImpute] Iter 6: observed MAE=0.145821 rank=52
[SoftImpute] Iter 7: observed MAE=0.145592 rank=52
[SoftImpute] Iter 8: observed MAE=0.145420 rank=51
[SoftImpute] Iter 9: observed MAE=0.145276 rank=50
[SoftImpute] Iter 10: observed MAE=0.145165 rank=50
[SoftImpute] Iter 11: observed MAE=0.145082 rank=50
[SoftImpute] Iter 12: observed MAE=0.145020 rank=50
[SoftImpute] Iter 13: observed MAE=0.144974 rank=50
[SoftImpute] Iter 14: observed MAE=0.144939 rank=50
[SoftImpute] Iter 15: observed MAE=0.144913 rank=50
[SoftImpute] Iter 16: observed MAE=0.144895 rank=50
[SoftImpute] Iter 17: observed MAE=0.144881 rank=50
[SoftImpute] Iter 18: observed MAE=0.144872 rank=50
[SoftImpute] Iter 

c:\Users\Chi\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\Chi\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\Chi\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


[SoftImpute] Max Singular Value of X_init = 545.577296
[SoftImpute] Iter 1: observed MAE=0.177247 rank=125
[SoftImpute] Iter 2: observed MAE=0.175449 rank=117
[SoftImpute] Iter 3: observed MAE=0.174372 rank=114
[SoftImpute] Iter 4: observed MAE=0.173731 rank=112
[SoftImpute] Iter 5: observed MAE=0.173325 rank=110
[SoftImpute] Iter 6: observed MAE=0.173059 rank=110
[SoftImpute] Iter 7: observed MAE=0.172878 rank=110
[SoftImpute] Iter 8: observed MAE=0.172750 rank=110
[SoftImpute] Iter 9: observed MAE=0.172657 rank=110
[SoftImpute] Iter 10: observed MAE=0.172587 rank=110
[SoftImpute] Iter 11: observed MAE=0.172534 rank=109
[SoftImpute] Iter 12: observed MAE=0.172494 rank=109
[SoftImpute] Iter 13: observed MAE=0.172463 rank=109
[SoftImpute] Iter 14: observed MAE=0.172439 rank=109
[SoftImpute] Iter 15: observed MAE=0.172420 rank=109
[SoftImpute] Iter 16: observed MAE=0.172405 rank=109
[SoftImpute] Iter 17: observed MAE=0.172392 rank=109
[SoftImpute] Iter 18: observed MAE=0.172382 rank=109


c:\Users\Chi\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\Chi\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\Chi\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


...

[SoftImpute] Max Singular Value of X_init = 857.926004
[SoftImpute] Iter 1: observed MAE=0.181795 rank=89
[SoftImpute] Iter 2: observed MAE=0.179304 rank=81
[SoftImpute] Iter 3: observed MAE=0.178363 rank=78
[SoftImpute] Iter 4: observed MAE=0.177896 rank=77
[SoftImpute] Iter 5: observed MAE=0.177621 rank=75
[SoftImpute] Iter 6: observed MAE=0.177442 rank=75
[SoftImpute] Iter 7: observed MAE=0.177319 rank=74
[SoftImpute] Iter 8: observed MAE=0.177233 rank=73
[SoftImpute] Iter 9: observed MAE=0.177174 rank=73
[SoftImpute] Iter 10: observed MAE=0.177135 rank=73
[SoftImpute] Iter 11: observed MAE=0.177110 rank=73
[SoftImpute] Iter 12: observed MAE=0.177096 rank=73
[SoftImpute] Iter 13: observed MAE=0.177090 rank=73
[SoftImpute] Iter 14: observed MAE=0.177089 rank=73
[SoftImpute] Iter 15: observed MAE=0.177093 rank=73
[SoftImpute] Iter 16: observed MAE=0.177100 rank=73
[SoftImpute] Iter 17: observed MAE=0.177108 rank=73
[SoftImpute] Iter 18: observed MAE=0.177119 rank=73
[SoftImpute] Iter 

c:\Users\Chi\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\Chi\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\Chi\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


[SoftImpute] Max Singular Value of X_init = 941.610187
[SoftImpute] Iter 1: observed MAE=0.192406 rank=71
[SoftImpute] Iter 2: observed MAE=0.189784 rank=65
[SoftImpute] Iter 3: observed MAE=0.188841 rank=62
[SoftImpute] Iter 4: observed MAE=0.188362 rank=60
[SoftImpute] Iter 5: observed MAE=0.188074 rank=58
[SoftImpute] Iter 6: observed MAE=0.187884 rank=58
[SoftImpute] Iter 7: observed MAE=0.187757 rank=58
[SoftImpute] Iter 8: observed MAE=0.187669 rank=57
[SoftImpute] Iter 9: observed MAE=0.187610 rank=57
[SoftImpute] Iter 10: observed MAE=0.187571 rank=57
[SoftImpute] Iter 11: observed MAE=0.187546 rank=57
[SoftImpute] Iter 12: observed MAE=0.187533 rank=57
[SoftImpute] Iter 13: observed MAE=0.187527 rank=57
[SoftImpute] Iter 14: observed MAE=0.187528 rank=57
[SoftImpute] Iter 15: observed MAE=0.187533 rank=57
[SoftImpute] Iter 16: observed MAE=0.187541 rank=57
[SoftImpute] Iter 17: observed MAE=0.187551 rank=57
[SoftImpute] Iter 18: observed MAE=0.187563 rank=57
[SoftImpute] Iter 

c:\Users\Chi\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\Chi\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\Chi\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


[SoftImpute] Max Singular Value of X_init = 1024.832956
[SoftImpute] Iter 1: observed MAE=0.203554 rank=62
[SoftImpute] Iter 2: observed MAE=0.200420 rank=56
[SoftImpute] Iter 3: observed MAE=0.199343 rank=54
[SoftImpute] Iter 4: observed MAE=0.198824 rank=52
[SoftImpute] Iter 5: observed MAE=0.198508 rank=52
[SoftImpute] Iter 6: observed MAE=0.198297 rank=51
[SoftImpute] Iter 7: observed MAE=0.198148 rank=50
[SoftImpute] Iter 8: observed MAE=0.198044 rank=50
[SoftImpute] Iter 9: observed MAE=0.197973 rank=50
[SoftImpute] Iter 10: observed MAE=0.197924 rank=50
[SoftImpute] Iter 11: observed MAE=0.197892 rank=50
[SoftImpute] Iter 12: observed MAE=0.197873 rank=50
[SoftImpute] Iter 13: observed MAE=0.197863 rank=50
[SoftImpute] Iter 14: observed MAE=0.197860 rank=50
[SoftImpute] Iter 15: observed MAE=0.197862 rank=50
[SoftImpute] Iter 16: observed MAE=0.197867 rank=50
[SoftImpute] Iter 17: observed MAE=0.197876 rank=50
[SoftImpute] Iter 18: observed MAE=0.197886 rank=50
[SoftImpute] Iter